# Probability Theory and Probabilistic Models

This lab has three parts, each taking a different probabilistic tool to behavioral data.

**Part 1** fits a Poisson process to response timing and then checks whether the model's own
predictions hold. You will estimate a rate by maximum likelihood, put a confidence interval on it,
and run three diagnostics that can reject the model. Two records are supplied that return the same
rate estimate; only the diagnostics tell them apart.

**Part 2** applies Bayesian updating to functional analysis data. Starting from a uniform prior
over four candidate functions, you will update after every session and watch the posterior move.
You will run the procedure on a clear case and on an ambiguous one, which behave very differently.

**Part 3** uses Monte Carlo simulation to put a confidence interval on a behavioral parameter
without an analytic formula.

## Background

A standard functional analysis (FA; Iwata et al., 1982/1994) arranges conditions -- here attention,
escape, tangible, and play (control) -- to identify the maintaining variable for problem behavior.
Clinicians typically rely on visual analysis. A Bayesian approach offers a quantitative complement:
state what each candidate function predicts, then let the data move a probability distribution over
those candidates.

Throughout, note that a probabilistic model specifies more than a mean. It specifies a whole
distribution, so the data can disagree with it in more than one way.

## Task 1: Import Libraries

Import `pandas`, `numpy`, `matplotlib.pyplot`, and `scipy.stats`. Set a random seed so your
Monte Carlo results in Part 3 are reproducible.

In [ ]:
# Import the libraries you need and set a random seed

---
# Part 1: The Poisson Process as a Model of Response Timing

`operant_irt_data.csv` holds two 50-minute records, `M-01` and `M-02`. Each row is one response,
identified by the subject and the time in seconds from the start of the session. Everything else
you need you will derive.

## Task 2: Load the Operant Data and Derive Inter-Response Times

Load `operant_irt_data.csv`. For each subject, compute:

- the vector of inter-response times (IRTs), which are the differences between successive response
  times;
- the number of responses in each successive 1-minute bin across the 50-minute session.

Report how many responses each subject emitted and the mean IRT for each. Before going further,
note whether the two records look similar on these summary numbers.

In [ ]:
# Load operant_irt_data.csv; derive IRTs and per-minute counts for each subject

## Task 3: Maximum Likelihood Estimate of the Rate

For a Poisson process observed over a fixed duration, the maximum likelihood estimate of the rate
is the total number of events divided by the total observation time:

$$\hat{\lambda} = \frac{N}{T}$$

Compute $\hat{\lambda}$ for each subject in responses per minute. Then compute the standard error
using $SE(\hat{\lambda}) = \sqrt{\hat{\lambda}/n}$, where $n$ is the number of 1-minute bins,
and form an approximate 95% confidence interval.

Do the two subjects differ in rate?

In [ ]:
# lambda-hat, SE, and a 95% CI for each subject

## Task 4: The Variance Check

The Poisson distribution has one parameter, so it makes a prediction with no second parameter to
absorb a mismatch: the variance of the counts equals their mean. Nothing was fitted to the
variance, so comparing the two is a test rather than a restatement of the fit.

For each subject, compute the mean and variance of the per-minute counts and their ratio. A ratio
near 1 is consistent with the model. A ratio well above 1 is overdispersion; well below 1 is
underdispersion. Which subject fails this check, and in which direction?

In [ ]:
# Compare the mean and variance of the per-minute counts for each subject

## Task 5: The Inter-Response Time Distribution

A Poisson process at rate $\lambda$ implies that IRTs follow an exponential distribution:

$$f(\tau) = \lambda e^{-\lambda \tau}$$

For each subject, plot a density histogram of the observed IRTs and overlay the exponential density
implied by that subject's own $\hat{\lambda}$. Work in seconds, so convert your rate from
responses per minute back to responses per second.

Where does each record depart from the prediction, and in which direction?

In [ ]:
# IRT density histogram per subject with the predicted exponential overlaid

## Task 6: Quantile-Quantile Plot

A histogram makes gross departures visible; a Q-Q plot makes the shape of the departure legible.
Plot the observed IRT quantiles against the theoretical quantiles of the exponential distribution
implied by each subject's $\hat{\lambda}$. Add the diagonal. Points on the diagonal indicate
agreement.

The theoretical quantile at probability $p$ for an exponential with rate $\lambda$ is
$-\ln(1 - p)/\lambda$.

Describe the shape of the departure for the subject that fails. What does a curve that sits below
the diagonal at short quantiles and rises above it at long quantiles tell you about the process?

In [ ]:
# Q-Q plot of observed IRT quantiles against exponential quantiles

## Task 7: What the Diagnostics Added

Both subjects returned nearly the same $\hat{\lambda}$ with overlapping confidence intervals.
Summarize, in a short printed statement or a markdown cell, what each of the three checks
(variance-to-mean, IRT histogram, Q-Q plot) revealed that the point estimate did not.

Then answer: if you had reported only the mean response rate for these two subjects, what would you
have missed, and would any behavioral conclusion have changed?

In [ ]:
# Summarize what the diagnostics revealed that the rate estimate did not

---
# Part 2: Bayesian Updating for Functional Assessment

Two FA datasets are supplied. Both cycle through attention, escape, tangible, and play in a fixed
order, and both carry the session `count`, the `duration_min`, and the resulting `rate_per_min`.

- `functional_analysis_data.csv` -- a clear case, 10-minute sessions.
- `functional_analysis_ambiguous.csv` -- a hard case, 5-minute sessions.

You will build the updating procedure on the clear case, then apply it to the ambiguous one.

## Task 8: Load and Inspect Both Datasets

Load both files. For each, report the number of sessions per condition and the mean rate per
condition. Then produce the standard FA graph for each: rate per minute on the y-axis against
session number on the x-axis, one connected series per condition.

For each dataset, write down the conclusion you would reach from visual analysis alone, and how
confident you would be.

In [ ]:
# Load both FA files, summarize by condition, and draw both FA graphs

## Task 9: Set Up the Prior

Define a uniform prior over four candidate functions: `attention`, `escape`, `tangible`, and
`automatic`. Store it as an array or dictionary and plot it as a bar chart.

Note what the fourth hypothesis is doing here. There is no "automatic" *condition* in an FA. What
distinguishes automatic reinforcement is that the behavior persists regardless of the social
consequence arranged, so it predicts elevated responding in *every* condition, including the play
control. Keep that in mind for the next task.

In [ ]:
# Build a uniform prior over the four candidate functions and plot it

## Task 10: Define the Likelihood

The likelihood is where the behavioral account enters. Each candidate function makes a claim about
which conditions should evoke elevated responding:

- `attention`, `escape`, and `tangible` predict elevation **only** in their matching condition.
- `automatic` predicts elevation in **every** condition, including play.
- In any condition where a function does not predict elevation, it predicts responding at a base
  rate.

Model the session count as Poisson. If a function predicts elevation for a session's condition, the
expected count is $\lambda_{\text{elevated}} \times \text{duration}$; otherwise it is
$\lambda_{\text{base}} \times \text{duration}$. Then

$$P(\text{count} \mid \text{function}) = \text{Poisson}(\text{count} \mid \lambda \times \text{duration})$$

Write a function returning the **log** likelihood for one session under one hypothesized function.
Work in logs throughout: the products in Bayes' theorem become sums, which is both numerically
safer and exactly the move the chapter makes when it switches to log-likelihood.

For the clear dataset, take $\lambda_{\text{base}} = 1.0$ and $\lambda_{\text{elevated}} = 9.0$
responses per minute. These are modeling assumptions, justified by the condition means you computed
in Task 8. You will test how much they matter in Task 13.

In [ ]:
# Define log_likelihood(function, condition, count, duration_min, base_rate, elevated_rate)

## Task 11: Implement the Updating Loop

Write a function that starts from the uniform prior and updates once per session, in session order,
returning the posterior after every session so you can plot the trajectory.

In log space, Bayes' theorem for each step is

$$\log p(\text{function} \mid \text{data}) = \log p(\text{function}) + \log P(\text{data} \mid \text{function}) - \log P(\text{data})$$

Rather than computing $\log P(\text{data})$ directly, accumulate the unnormalized log posterior
and normalize only when you need probabilities: subtract the maximum, exponentiate, and divide by
the sum. This avoids underflow when one hypothesis becomes overwhelmingly favored.

Run it on the **clear** dataset and print the posterior after each of the first four sessions.

Look carefully at what happens after session 1, and then at what session 2 does. Explain it.

In [ ]:
# Implement run_updating(df, base_rate, elevated_rate) and run it on the clear dataset

## Task 12: Plot the Posterior Trajectories for Both Datasets

Run the updating on both datasets and plot the posterior probability of each function against
session number, one panel per dataset. Use $\lambda_{\text{base}} = 2.0$ and
$\lambda_{\text{elevated}} = 2.6$ for the ambiguous dataset, again justified by its condition
means.

For each dataset report the first session at which the leading function exceeds 0.90 and the first
at which it exceeds 0.99.

Compare the two trajectories. Which one tells you something visual analysis could not?

In [ ]:
# Run updating on both datasets, plot the trajectories, report convergence sessions

## Task 13: Sensitivity to the Assumed Rates

$\lambda_{\text{base}}$ and $\lambda_{\text{elevated}}$ were assumptions, not estimates. A
conclusion that depends heavily on them is worth less than one that does not.

For each dataset, hold $\lambda_{\text{base}}$ fixed and sweep $\lambda_{\text{elevated}}$
across a range of plausible values. Record the final posterior probability of the true function for
each value, and plot or tabulate the result.

Which dataset's conclusion is robust to the assumption, and which is not? What does that imply
about reporting a Bayesian FA analysis?

In [ ]:
# Sweep elevated_rate for both datasets and record the final posterior

---
# Part 3: Monte Carlo Simulation for Confidence Intervals

Not every quantity you care about has a tidy standard error. Simulation gives you an interval
regardless, by resampling the data you have.

## Task 14: Bootstrap the Attention-Condition Mean

Using the **clear** FA dataset, estimate the mean rate in the attention condition and put a 95%
confidence interval on it by bootstrap:

1. Extract the attention-condition rates.
2. Draw 10,000 resamples, each the same size as the original, sampled **with replacement**.
3. Compute the mean of each resample.
4. Take the 2.5th and 97.5th percentiles of those 10,000 means.
5. Plot the bootstrap distribution with the observed mean and both interval bounds marked.

Then compute the ordinary $t$-based interval on the same data and compare. Where do they differ,
and which assumption is each one making?

In [ ]:
# Bootstrap the attention-condition mean, then compare against the t interval

## Wrap-up

Answer the following in a markdown cell. These are the points the lab was built around.

1. Both subjects in Part 1 returned the same rate estimate. What is the general lesson about
   reporting a fitted parameter without reporting the checks that could have rejected the model?
2. In Part 2, the posterior after a single elevated attention session was split evenly between
   `attention` and `automatic`. What does that say about which sessions in an FA carry the
   discriminating evidence?
3. What are the advantages of Bayesian updating over visual analysis for FA interpretation, and
   what are its limitations? Be specific about what the prior and the likelihood are doing.
4. Under what circumstances would you trust the ambiguous dataset's final posterior of 0.9998, and
   under what circumstances would you not?
5. Where might a probabilistic approach to functional assessment be most valuable in practice?

In [ ]:
# Write your answers in a markdown cell below